# Dataset Visualization

Visual overview of every GEO series-matrix dataset configured in
`configs/datasets.yaml` whose CSV exists on disk (built via notebook 00).

**Rare-class threshold `K` is configurable (section 1).** Defaults to
`K=0` — origin data, no filtering, so it's the one place to see what a class
distribution looks like *before* any min_samples_per_class threshold removes
anything. Set `K>0` to preview the same threshold notebooks 02/04 sweep
(`feature_selection.yaml -> rare_class_k_values`) — outputs then go to
`outputs/visualizations_k{K}/` instead of `outputs/visualizations/`.

Each dataset produces a 4-panel figure:
| Panel | Description |
|-------|-------------|
| Class distribution | Sample counts per class |
| PCA | 2-D projection (PC1 vs PC2, coloured by class) |
| t-SNE | Non-linear 2-D embedding (coloured by class) |
| Expression heatmap | Top-N variable probes × samples (z-score, sorted by class) |


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ directory.
REPO_ROOT = Path("..").resolve()              # repo root (parent of notebooks/)
sys.path.insert(0, str(REPO_ROOT))            # make src.* importable

%matplotlib inline
import matplotlib
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "font.family": "sans-serif",
    "axes.grid": False,
})

In [ ]:
# Rare-class threshold for this notebook's plots/exports.
# K=0 (default)  -> origin data, no filtering (see intro above).
# K=1 (just same result with k=0)
# K>1            -> drop classes with < K samples before plotting, same
#                   min_samples_per_class threshold notebooks 02/04 sweep
#                   (feature_selection.yaml -> rare_class_k_values). Output
#                   then goes to outputs/visualizations_k{K}/ instead of
#                   outputs/visualizations/, so different K's never overwrite
#                   each other.
K = 6


## 2. Collect datasets from config

Datasets are read from `configs/datasets.yaml` (any entry whose CSV exists on
disk), so this notebook tracks the single source of truth — no hardcoded paths.

In [ ]:
import pandas as pd
from src.helper import ConfigLoader

OUTPUT_DIR  = REPO_ROOT / "outputs" / ("visualizations" if K == 0 else f"visualizations_k{K}")
ORIG_DIR    = OUTPUT_DIR / "original"   # plots with class names only
LABELED_DIR = OUTPUT_DIR / "labeled"    # plots with 'name (id)' labels
for d in (OUTPUT_DIR, ORIG_DIR, LABELED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── collect datasets from configs/datasets.yaml whose CSV exists on disk ──
# Expand every registry entry by its `levels` (probe / gene) into run-keys,
# regardless of `enabled`, so this notebook can preview any built CSV.
cfg = ConfigLoader(str(REPO_ROOT / "configs"))
datasets_cfg = cfg.load_datasets_config()["datasets"]

run_keys = []
for name, dc in datasets_cfg.items():
    for level in dc.get("levels", ["probe"]):
        run_keys.append(name if level == "probe" else f"{name}-{level}")

all_paths    = []      # list[Path] to each dataset CSV
name_by_path = {}      # Path -> config dataset name
rows = []
for ds_name in run_keys:
    dc = cfg.get_dataset_config(ds_name)
    p = Path(dc["path"])
    if p.exists():
        all_paths.append(p)
        name_by_path[p] = ds_name
        rows.append({
            "dataset": ds_name,
            "enabled": dc.get("enabled", False),
            "file": p.stem,
            "path": str(p.relative_to(REPO_ROOT)),
        })

if not all_paths:
    print("No dataset CSVs found on disk. Run notebook 00.a (dataset builder) first.")
pd.DataFrame(rows)

## 3. Dataset summary statistics

In [ ]:
from src.helper import load_dataset
from src.visualize import class_names

summary_rows = []
loaded = {}  # cache loaded datasets so we don't reload in section 4

for p in all_paths:
    data = load_dataset(p, dataset_type="cumida", min_samples_per_class=K)  # K from section 1 (default 0 = origin data, no filtering)
    loaded[p] = data
    m = data["metadata"]
    summary_rows.append({
        "dataset":  name_by_path[p],
        "file":     p.stem,
        "samples":  m["n_samples"],
        "features": m["n_features"],
        "classes":  m["n_classes"],
        "class_names": ", ".join(class_names(data)),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 4. Individual plots

Use the cells below to inspect specific plot types for any dataset.

In [ ]:
# ── pick one dataset by file name ─────────────────────────────────────────
from src.visualize import (
    plot_class_distribution,
    plot_pca,
    plot_tsne,
    plot_expression_heatmap,
)

target_name = "Breast_GSE20711_probe"   # <-- any 'file' value from the preview table above
target_path = next((p for p in all_paths if p.stem == target_name), None)

if target_path is None:
    target_path = all_paths[0] if all_paths else None
    if target_path is not None:
        print(f"'{target_name}' not found — defaulting to '{target_path.stem}'.")
        print("Available:", sorted(p.stem for p in all_paths))

if target_path is not None:
    data = loaded[target_path]
    print(f"Dataset : {name_by_path[target_path]} ({target_path.stem})")
    print(f"Samples : {data['metadata']['n_samples']}")
    print(f"Features: {data['metadata']['n_features']}")
    print(f"Classes : {data['metadata']['n_classes']}")
else:
    print("No datasets available to plot.")

In [ ]:
if target_path is not None:
    plot_class_distribution(data, title=f"{target_name} – class distribution");

In [ ]:
if target_path is not None:
    plot_pca(data, title=f"{target_name} – PCA");

In [ ]:
if target_path is not None:
    plot_tsne(data, title=f"{target_name} – t-SNE");

In [ ]:
if target_path is not None:
    plot_expression_heatmap(data, title=f"{target_name} – heatmap", top_n=50);

## 5. Full overview — all datasets

Generates one 4-panel figure per dataset and saves two copies:

| Folder | Description |
|--------|-------------|
| `outputs/visualizations/original/` | Class names only (e.g. *basal*) |
| `outputs/visualizations/labeled/` | Class name + integer ID (e.g. *basal (0)*) |

In [ ]:
from src.visualize import plot_dataset_overview
import matplotlib.pyplot as plt

for p in all_paths:
    data = loaded[p]
    name = p.stem
    print(f"Plotting {name} ...")

    for show_ids, out_dir, tag in (
        (False, ORIG_DIR,    "original"),
        (True,  LABELED_DIR, "labeled"),
    ):
        fig = plot_dataset_overview(
            data, dataset_name=name,
            figsize=(16, 12), top_n_heatmap=50,
            show_class_ids=show_ids,
        )
        out_path = out_dir / f"{name}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"  [{tag}] -> {out_path.relative_to(REPO_ROOT)}")

        if show_ids:
            plt.show()   # show the labeled version inline (more informative)
        plt.close(fig)

print(f"\nDone. Figures saved under {OUTPUT_DIR.relative_to(REPO_ROOT)}")

## 6. Cross-dataset comparison — class balance

Side-by-side view of class imbalance across all datasets.

In [ ]:
import matplotlib.pyplot as plt
from src.visualize import plot_class_distribution

n_datasets = len(all_paths)
ncols = 4
nrows = (n_datasets + ncols - 1) // ncols

for show_ids, out_dir, fname in (
    (False, ORIG_DIR,    "_all_class_distributions.png"),
    (True,  LABELED_DIR, "_all_class_distributions.png"),
):
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3.5))
    axes_flat = axes.flatten()

    for i, p in enumerate(all_paths):
        plot_class_distribution(loaded[p], title=p.stem, ax=axes_flat[i],
                                show_class_ids=show_ids)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    label = "with class IDs" if show_ids else "original"
    fig.suptitle(f"Class distribution — all datasets ({label})",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()

    out_path = out_dir / fname
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path.relative_to(REPO_ROOT)}")
    plt.show()
    plt.close(fig)

## 7. Cross-dataset comparison — PCA

In [ ]:
from src.visualize import plot_pca

for show_ids, out_dir, fname in (
    (False, ORIG_DIR,    "_all_pca.png"),
    (True,  LABELED_DIR, "_all_pca.png"),
):
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes_flat = axes.flatten()

    for i, p in enumerate(all_paths):
        plot_pca(loaded[p], title=p.stem, ax=axes_flat[i],
                 show_class_ids=show_ids)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    label = "with class IDs" if show_ids else "original"
    fig.suptitle(f"PCA — all datasets ({label})",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()

    out_path = out_dir / fname
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out_path.relative_to(REPO_ROOT)}")
    plt.show()
    plt.close(fig)